# Notebook 03 — External Data (Weather, Events, Holidays)

This notebook runs the actual download/clean pipelines implemented in
`src/external_weather.py` and `src/external_events.py` (the same code
`python -m src.external_weather` / `python -m src.external_events` run), then
surfaces the weather/events/holiday rows of **Table 1** (dataset sizes).

Event geocoding reuses a committed lookup cache (`data/external/events/geocode_cache.json`)
rather than re-querying the public Nominatim API for all ~4,325 unique event locations from
scratch (~80 minutes, rate-limited to 1 request/1.1s) — see `.gitignore` for why this one
cache file is committed rather than regenerated on every run.

**Note on scope:** Section 2.2's external-join-coverage percentages (99.76% weather /
26.76% events / 2.95% holidays) are computed against the zone-hour grid, which doesn't
exist until the Processed stage — they're surfaced in notebook `04`, not here.

## 1. Weather — download + clean

Source: Iowa Environmental Mesonet ASOS archive, station LGA.

In [1]:
from src.external_weather import (
    CLEAN_CSV_PATH as WEATHER_CLEAN_CSV_PATH,
    clean_weather,
    download_weather,
    load_raw_weather,
    profile_weather,
    write_weather_report,
)

download_weather()
raw_weather = load_raw_weather()
clean_weather_df = clean_weather(raw_weather)

print(f"Weather raw rows:   {len(raw_weather):,} ({raw_weather.shape[1]} columns)")
print(f"Weather clean rows: {len(clean_weather_df):,}")

2026-08-28 16:11:26.387 | INFO     | src.config:<module>:26 - PROJ_ROOT path is: /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final


2026-08-28 16:11:26.818 | INFO     | src.external_weather:download_weather:73 - Downloading LGA hourly weather 2024-01-01 -> 2026-06-01 from IEM ASOS


2026-08-28 16:11:34.410 | SUCCESS  | src.external_weather:download_weather:81 - Wrote 21,122 rows to /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/external/weather/lga_hourly_raw.csv


Weather raw rows:   21,122 (11 columns)
Weather clean rows: 21,118


In [2]:
WEATHER_CLEAN_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_weather_df.to_csv(WEATHER_CLEAN_CSV_PATH, index=False)

weather_report = profile_weather(raw_weather, clean_weather_df)
write_weather_report(weather_report)
print(f"Weather severity distribution: {weather_report['severity_distribution']}")

2026-08-28 16:11:34.695 | INFO     | src.external_weather:write_weather_report:248 - Weather data-quality report written to /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/reports/data_quality/external/weather_report.json
Weather severity distribution: {'normal': 17142, 'mild': 3238, 'severe': 738}


## 2. Events — download, clean, geocode

Source: NYC Open Data, "NYC Permitted Event Information" (Socrata `bkfu-528j`).

In [3]:
from src.external_events import (
    CLEAN_CSV_PATH as EVENTS_CLEAN_CSV_PATH,
    GEOCODE_CACHE_PATH,
    assign_taxi_zones,
    build_location_keys,
    clean_events,
    download_events,
    geocode_locations,
    load_raw_events,
    write_events_report,
)

download_events()
raw_events = load_raw_events()
clean_events_df, events_report = clean_events(raw_events)

print(f"Events raw rows:   {len(raw_events):,} ({raw_events.shape[1]} columns)")
print(f"Events clean rows: {len(clean_events_df):,}")

2026-08-28 16:11:34.840 | INFO     | src.external_events:download_events:79 - Fetching events offset=0


2026-08-28 16:11:41.722 | INFO     | src.external_events:download_events:79 - Fetching events offset=50000


2026-08-28 16:11:48.886 | INFO     | src.external_events:download_events:79 - Fetching events offset=100000


2026-08-28 16:11:52.312 | INFO     | src.external_events:download_events:79 - Fetching events offset=150000


2026-08-28 16:12:19.626 | INFO     | src.external_events:download_events:79 - Fetching events offset=200000


...

2026-08-28 16:13:07.438 | INFO     | src.external_events:download_events:79 - Fetching events offset=500000


2026-08-28 16:13:13.312 | INFO     | src.external_events:download_events:79 - Fetching events offset=550000


2026-08-28 16:13:20.845 | INFO     | src.external_events:download_events:79 - Fetching events offset=600000


2026-08-28 16:13:26.888 | SUCCESS  | src.external_events:download_events:93 - Wrote 629,145 rows to /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/external/events/nyc_permitted_events_raw.csv


Events raw rows:   629,145 (13 columns)
Events clean rows: 612,656


Geocoding — resumable and disk-cached; only locations not already in the committed cache trigger a live Nominatim call.

In [4]:
keys = build_location_keys(clean_events_df)
print(f"Unique (location, borough) keys: {len(keys):,}")

geocode_locations(keys, cache_path=GEOCODE_CACHE_PATH)
clean_events_df = assign_taxi_zones(clean_events_df, cache_path=GEOCODE_CACHE_PATH)

events_report["geocoding"] = {
    "unique_location_keys": len(keys),
    "events_geocoded": int(clean_events_df["pu_location_id"].notna().sum()),
    "events_not_geocoded": int(clean_events_df["pu_location_id"].isna().sum()),
    "pct_geocoded": round(100 * clean_events_df["pu_location_id"].notna().mean(), 2),
}
print(events_report["geocoding"])

Unique (location, borough) keys: 4,325
2026-08-28 16:13:28.531 | SUCCESS  | src.external_events:geocode_locations:203 - Geocoding pass: 0 new (0 failed), 4325 already cached


{'unique_location_keys': 4325, 'events_geocoded': 581346, 'events_not_geocoded': 31310, 'pct_geocoded': np.float64(94.89)}


In [5]:
EVENTS_CLEAN_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_events_df.to_csv(EVENTS_CLEAN_CSV_PATH, index=False)
write_events_report(events_report)
print("Events clean dataset + report written.")

2026-08-28 16:13:34.193 | INFO     | src.external_events:write_events_report:423 - Events data-quality report written to /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/reports/data_quality/external/events_report.json
Events clean dataset + report written.


## 3. Holidays — dataset size

A small, manually-curated CSV (no download/regeneration script), committed directly.

In [6]:
import pandas as pd

from src.config import EXTERNAL_DATA_DIR

holidays = pd.read_csv(EXTERNAL_DATA_DIR / "holiday_schedule.csv")
print(f"Holidays rows: {len(holidays):,} ({holidays.shape[1]} columns)")
holidays

Holidays rows: 34 (2 columns)


,Date,Holiday
0,1/1/24,New Year's Day
1,1/15/24,"Birthday of Martin Luther King, Jr."
2,2/19/24,Washington's Birthday
3,5/27/24,Memorial Day
4,6/19/24,Juneteenth National Independence Day
5,7/4/24,Independence Day
6,9/2/24,Labor Day
7,10/14/24,Columbus Day
8,11/11/24,Veterans Day
9,11/28/24,Thanksgiving Day


## 4. Table 1 — external dataset sizes

Row and column counts for each of the three external datasets, at their raw/source
stage.

In [7]:
print("Table 1 -- external datasets")
print(f"  Hourly weather (LGA):  {len(raw_weather):,} rows, {raw_weather.shape[1]} features")
print(f"  NYC permitted events:  {len(raw_events):,} rows, {raw_events.shape[1]} features")
print(f"  US federal holidays:   {len(holidays):,} rows, {holidays.shape[1]} features")

Table 1 -- external datasets
  Hourly weather (LGA):  21,122 rows, 11 features
  NYC permitted events:  629,145 rows, 13 features
  US federal holidays:   34 rows, 2 features


## Notes

- Output: `data/external/weather/lga_hourly_clean.csv`,
  `data/external/events/nyc_permitted_events_clean.csv`, plus data-quality reports under
  `reports/data_quality/external/`.
- Section 2.2's join-coverage percentages (weather/events/holidays against the zone-hour
  grid) are computed in notebook `04`, which builds that grid.